# BirdCLEF+ 2026 Perch v2 Probe

Extract **frozen Google Perch v2 embeddings** and train a shallow PyTorch probe for the 206 BirdCLEF+ 2026 primary labels. Use **`CFG.mode = "train"`** for the full experiment and **`CFG.mode = "submission"`** to load the uploaded probe artifact and score test soundscapes.


## 1. Environment

Configure the Kaggle runtime, attach the **TensorFlow 2.20 wheel input**, and load the libraries used for Perch embedding extraction and PyTorch probe training.


In [1]:
from pathlib import Path
import json
import os
import random
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


class CFG:
    """Base configuration for the Perch v2 probe notebook."""

    seed = 42
    competition_name = "birdclef-2026"
    data_root = None
    artifact_dir = Path("/kaggle/working/artifacts")
    tensorflow_wheel_root = Path("/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0")
    perch_input_roots = [
        Path("/kaggle/input/datasets/jaejohn/perch-meta"),
        Path(
            "/kaggle/input/models/google/"
            "bird-vocalization-classifier/tensorflow2/perch_v2/2"
        ),
        Path("/kaggle/input/perch-meta"),
    ]
    perch_artifact_roots = [
        Path(
            "/kaggle/input/models/tuannm3812/"
            "birdclef-perch-v2-artifacts/pytorch/default/1/perch_v2"
        ),
    ]


def seed_everything(seed: int = 42) -> None:
    """Set deterministic seeds for Python and NumPy.

    Args:
        seed (int): Seed used for reproducible notebook operations.
    """
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def find_data_root() -> Path:
    """Find the attached BirdCLEF+ 2026 Kaggle dataset root.

    Returns:
        Path: Directory containing `train.csv`.

    Raises:
        FileNotFoundError: If no candidate dataset directory is found.
    """
    candidates = [
        Path("/kaggle/input/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack"),
    ]
    for path in candidates:
        if (path / "train.csv").exists():
            return path
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = list(input_root.glob("**/train.csv"))
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not find train.csv. Attach the BirdCLEF 2026 dataset.")


def package_version(package_name: str) -> str | None:
    """Return an installed package version when available.

    Args:
        package_name (str): Distribution name.

    Returns:
        str | None: Installed version string or `None`.
    """
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


def version_tuple(value: str) -> tuple[int, ...]:
    """Convert a version string into comparable integer components.

    Args:
        value (str): Version string such as `2.20.0`.

    Returns:
        tuple[int, ...]: Numeric version prefix.
    """
    core = value.split("+")[0].split("-")[0]
    parts = []
    for part in core.split("."):
        if part.isdigit():
            parts.append(int(part))
        else:
            break
    return tuple(parts)


def install_tensorflow_220() -> None:
    """Install TensorFlow 2.20 from the attached Kaggle wheel input.

    Raises:
        FileNotFoundError: If the TensorFlow wheel cannot be found.
    """
    tf_wheels = sorted(CFG.tensorflow_wheel_root.glob("**/tensorflow-2.20.0*.whl"))
    tb_wheels = sorted(CFG.tensorflow_wheel_root.glob("**/tensorboard-2.20.0*.whl"))
    if not tf_wheels:
        raise FileNotFoundError(
            f"TensorFlow 2.20 wheel not found under {CFG.tensorflow_wheel_root}. "
            "Attach the bc26-tensorflow-2-20-0 Kaggle input before running notebook 3."
        )
    targets = [str(path) for path in tb_wheels[:1] + tf_wheels[:1]]
    print(f"Installing TensorFlow 2.20 from {CFG.tensorflow_wheel_root}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *targets], check=True)


seed_everything(CFG.seed)
CFG.data_root = find_data_root()
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Data root: {CFG.data_root}")
print(f"Output directory: {CFG.artifact_dir}")

import librosa
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

installed_tf = package_version("tensorflow")
if installed_tf is None or version_tuple(installed_tf) < version_tuple("2.20.0"):
    print(
        f"TensorFlow {installed_tf} is older than the Perch v2 requirement; "
        "installing 2.20.0 from Kaggle input."
    )
    install_tensorflow_220()

try:
    import tensorflow as tf
except ImportError:
    tf = None

torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)


class CFG(CFG):
    """Perch-specific configuration values."""

    mode = "submission"
    artifact_dir = Path("/kaggle/working/artifacts/perch_v2")
    submission_artifact_dir = None
    sample_rate = 32000
    duration = 5.0
    embedding_dim = 1536
    extraction_batch_size = 8
    probe_batch_size = 128
    probe_epochs = 20
    early_stopping_patience = 4
    early_stopping_min_delta = 1e-4
    hidden_dim = 512
    dropout = 0.25
    lr = 1e-3
    max_samples = None
    perch_model_dir = None
    reuse_attached_artifacts = True
    inference_batch_size = 8


CFG.artifact_dir.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Mode: {CFG.mode}")
if tf is not None:
    print(f"TensorFlow: {tf.__version__}")


Data root: /kaggle/input/competitions/birdclef-2026
Output directory: /kaggle/working/artifacts
TensorFlow 2.19.0 is older than the Perch v2 requirement; installing 2.20.0 from Kaggle input.
Installing TensorFlow 2.20 from /kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0
Device: cuda
Mode: submission
TensorFlow: 2.20.0


## 2. Metadata


Use the same **206 primary labels** as the EfficientNet baseline. In training mode, labels are written to the artifact directory. In submission mode, labels are loaded from the attached Perch probe artifact when available.


In [2]:
train = pd.read_csv(CFG.data_root / "train.csv")
train["filepath"] = train["filename"].map(lambda x: CFG.data_root / "train_audio" / x)
if CFG.mode == "train" and CFG.max_samples:
    train = train.sample(CFG.max_samples, random_state=CFG.seed).reset_index(drop=True)

labels = sorted(train["primary_label"].unique())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
train["target"] = train["primary_label"].map(label_to_idx)

if CFG.mode == "train":
    (CFG.artifact_dir / "labels.json").write_text(
        json.dumps(idx_to_label, indent=2),
        encoding="utf-8",
    )

print(f"Rows: {len(train):,}")
print(f"Classes: {len(labels):,}")
display(train.head())


Rows: 35,549
Classes: 206


,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection,filepath,target
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat,/kaggle/input/competitions/birdclef-2026/train...,0


## 3. Load Perch v2


Load the **Perch v2 TensorFlow SavedModel** and run a one-batch smoke test. This confirms the input signature, embedding shape, and TensorFlow/XLA compatibility before the full audio pass.


In [3]:
def find_perch_model_dir() -> Path:
    """Find the attached Perch TensorFlow SavedModel directory.

    Returns:
        Path: Directory containing `saved_model.pb`.

    Raises:
        FileNotFoundError: If no Perch SavedModel is attached.
    """
    if CFG.perch_model_dir:
        path = Path(CFG.perch_model_dir)
        if (path / "saved_model.pb").exists():
            return path
        matches = list(path.glob("**/saved_model.pb")) if path.exists() else []
        if matches:
            return matches[0].parent

    search_roots = [path for path in CFG.perch_input_roots if path.exists()]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        search_roots.append(input_root)

    matches = []
    for root in dict.fromkeys(search_roots):
        if (root / "saved_model.pb").exists():
            matches.append(root / "saved_model.pb")
        matches.extend(root.glob("**/saved_model.pb"))
    matches = [path.parent for path in matches if "perch" in str(path).lower() or "vocal" in str(path).lower() or "bird" in str(path).lower()]
    if matches:
        return matches[0]
    raise FileNotFoundError(
        "Could not find a Perch SavedModel. Attach /kaggle/input/datasets/jaejohn/perch-meta, "
        "the Perch/vocalization-classifier Kaggle model, or set CFG.perch_model_dir."
    )


if tf is None:
    raise ImportError("TensorFlow is required for Perch embedding extraction.")

perch_model_dir = find_perch_model_dir()
perch = tf.saved_model.load(str(perch_model_dir))
infer = perch.signatures["serving_default"]
print(f"Perch model: {perch_model_dir}")
print(f"Inputs: {infer.structured_input_signature}")
print(f"Outputs: {infer.structured_outputs}")


def explain_perch_runtime_error(error: Exception) -> None:
    """Raise a clearer message for known Perch runtime failures.

    Args:
        error (Exception): Original TensorFlow exception.

    Raises:
        RuntimeError: For known TensorFlow/XLA incompatibility errors.
        Exception: Re-raises the original error otherwise.
    """
    message = str(error)
    if "XlaCallModuleOp with version 10 is not supported" in message:
        raise RuntimeError(
            "The attached Perch SavedModel requires a newer TensorFlow/XLA runtime than this Kaggle image. "
            "The setup cell should install tensorflow>=2.20 before TensorFlow is imported. Restart the "
            "Kaggle session after installation, then run the notebook from the top. If internet is disabled, "
            "attach a Kaggle dataset containing a compatible TensorFlow wheel or use an older Perch SavedModel export."
        ) from error
    raise error


def smoke_test_perch() -> None:
    """Run a one-batch inference pass through the Perch signature."""
    dummy = tf.zeros((1, int(CFG.sample_rate * CFG.duration)), dtype=tf.float32)
    _, keyword_specs = infer.structured_input_signature
    try:
        if keyword_specs:
            input_name = next(iter(keyword_specs))
            outputs = infer(**{input_name: dummy})
        else:
            outputs = infer(dummy)
    except Exception as error:
        explain_perch_runtime_error(error)
    print({name: tuple(value.shape) for name, value in outputs.items()})


smoke_test_perch()


I0000 00:00:1779484996.644240      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779484996.647657      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Perch model: /kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2
Inputs: ((), {'inputs': TensorSpec(shape=(None, 160000), dtype=tf.float32, name='inputs')})
Outputs: {'embedding': TensorSpec(shape=(None, 1536), dtype=tf.float32, name='embedding'), 'spatial_embedding': TensorSpec(shape=(None, 16, 4, 1536), dtype=tf.float32, name='spatial_embedding'), 'spectrogram': TensorSpec(shape=(None, 500, 128), dtype=tf.float32, name='spectrogram'), 'label': TensorSpec(shape=(None, 14795), dtype=tf.float32, name='label')}


2026-05-22 21:23:27.006310: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 21:23:27.149313: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 21:23:28.634242: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 21:23:28.774130: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-22 21:23:28.964798: E external/local_xla/xla/stream_

{'embedding': (1, 1536), 'spatial_embedding': (1, 16, 4, 1536), 'spectrogram': (1, 500, 128), 'label': (1, 14795)}


## 4. Embedding Extraction


Convert each **5-second waveform** into a **1,536-dimensional Perch embedding**. Training mode can reuse the uploaded `train_embeddings.npz`; submission mode skips train embeddings and extracts only the hidden-test windows needed for `submission.csv`.


In [4]:
def load_audio(path: Path) -> np.ndarray:
    """Load a fixed-duration mono waveform for Perch inference.

    Args:
        path (Path): Audio file path.

    Returns:
        np.ndarray: Mono waveform padded or trimmed to `CFG.duration`.
    """
    target_len = int(CFG.sample_rate * CFG.duration)
    y, _ = librosa.load(
        path,
        sr=CFG.sample_rate,
        mono=True,
        duration=CFG.duration,
    )
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def load_audio_segment(path: Path, end_time: float) -> np.ndarray:
    """Load one fixed-duration audio segment ending at `end_time`.

    Args:
        path (Path): Audio file path.
        end_time (float): Segment end time in seconds.

    Returns:
        np.ndarray: Mono waveform segment.
    """
    offset = max(0.0, float(end_time) - CFG.duration)
    target_len = int(CFG.sample_rate * CFG.duration)
    y, _ = librosa.load(
        path,
        sr=CFG.sample_rate,
        mono=True,
        offset=offset,
        duration=CFG.duration,
    )
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def find_perch_artifact_dir() -> Path | None:
    """Find the attached Perch artifact directory when available.

    Returns:
        Path | None: Directory containing `best_perch_probe.pt` and
        `labels.json`, or `None` when no artifact is attached.
    """
    required_files = ["best_perch_probe.pt", "labels.json"]
    candidates = []

    if CFG.submission_artifact_dir is not None:
        candidates.append(Path(CFG.submission_artifact_dir))
    candidates.extend(CFG.perch_artifact_roots)

    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(path.parent for path in input_root.rglob("best_perch_probe.pt"))

    seen = set()
    for root in candidates:
        root = Path(root)
        if root in seen or not root.exists():
            continue
        seen.add(root)
        if all((root / name).exists() for name in required_files):
            return root

    return None

def load_embedding_cache(path: Path) -> np.ndarray:
    """Load cached Perch embeddings from an `.npz` file.

    Args:
        path (Path): Embedding cache path.

    Returns:
        np.ndarray: Cached embedding matrix.
    """
    saved = np.load(path, allow_pickle=True)
    return saved["embeddings"].astype(np.float32)


def run_perch_batch(batch_waveforms: np.ndarray) -> np.ndarray:
    """Extract Perch embeddings for a batch of waveforms.

    Args:
        batch_waveforms (np.ndarray): Batch shaped `(batch, samples)`.

    Returns:
        np.ndarray: Batch of 2D Perch embeddings.
    """
    tensor = tf.convert_to_tensor(batch_waveforms, dtype=tf.float32)
    _, keyword_specs = infer.structured_input_signature
    if keyword_specs:
        input_name = next(iter(keyword_specs))
        try:
            outputs = infer(**{input_name: tensor})
        except Exception as error:
            explain_perch_runtime_error(error)
    else:
        try:
            outputs = infer(tensor)
        except Exception as error:
            explain_perch_runtime_error(error)
    arrays = {name: np.asarray(value) for name, value in outputs.items()}
    embedding_name = next(
        (
            name
            for name in arrays
            if "embedding" in name.lower() or "embed" in name.lower()
        ),
        next(iter(arrays)),
    )
    value = arrays[embedding_name]
    if value.ndim == 3:
        value = value.mean(axis=1)
    elif value.ndim > 3:
        value = value.reshape(value.shape[0], -1)
    return value.astype(np.float32)


attached_artifact_dir = find_perch_artifact_dir()
print(f"Attached artifact directory: {attached_artifact_dir}")

if CFG.mode == "train":
    embeddings_path = CFG.artifact_dir / "train_embeddings.npz"
    attached_embeddings_path = None
    if attached_artifact_dir is not None:
        attached_embeddings_path = attached_artifact_dir / "train_embeddings.npz"

    if embeddings_path.exists():
        embeddings = load_embedding_cache(embeddings_path)
        print(f"Loaded embeddings from {embeddings_path}")
    elif CFG.reuse_attached_artifacts and attached_embeddings_path is not None:
        embeddings = load_embedding_cache(attached_embeddings_path)
        print(f"Loaded embeddings from {attached_embeddings_path}")
    else:
        chunks = []
        waveforms = []
        for path in tqdm(train["filepath"], desc="audio"):
            waveforms.append(load_audio(path))
            if len(waveforms) == CFG.extraction_batch_size:
                chunks.append(run_perch_batch(np.stack(waveforms)))
                waveforms = []
        if waveforms:
            chunks.append(run_perch_batch(np.stack(waveforms)))
        if not chunks:
            raise RuntimeError("No training embeddings were generated.")
        embeddings = np.concatenate(chunks, axis=0)
        np.savez_compressed(
            embeddings_path,
            embeddings=embeddings,
            labels=train["primary_label"].to_numpy(),
            filenames=train["filename"].to_numpy(),
        )
        print(f"Saved embeddings to {embeddings_path}")

    print(f"Embeddings: {embeddings.shape}")
else:
    print("Submission mode: skipped train embedding extraction.")


Attached artifact directory: /kaggle/input/models/tuannm3812/birdclef-perch-v2-artifacts/pytorch/default/1/perch_v2
Submission mode: skipped train embedding extraction.


## 5. Probe Model


The probe is deliberately shallow: **LayerNorm -> Linear -> ReLU -> Dropout -> Linear**. Perch should already encode most acoustic structure, so the classifier only needs to learn compact class boundaries on frozen features.


In [5]:
class PerchProbe(nn.Module):
    """Shallow classifier trained on frozen Perch embeddings."""

    def __init__(self, embedding_dim: int, num_classes: int) -> None:
        """Initialize the probe network.

        Args:
            embedding_dim (int): Width of the Perch embedding vector.
            num_classes (int): Number of target classes.
        """
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, CFG.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(CFG.dropout),
            nn.Linear(CFG.hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute class logits for a batch of embeddings.

        Args:
            x (torch.Tensor): Embedding tensor.

        Returns:
            torch.Tensor: Class logits.
        """
        return self.net(x)


def torch_load_checkpoint(path: Path) -> dict[str, object]:
    """Load a PyTorch checkpoint with Kaggle-version compatibility.

    Args:
        path (Path): Checkpoint path.

    Returns:
        dict[str, object]: Loaded checkpoint dictionary.
    """
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def load_artifact_labels(artifact_dir: Path) -> list[str]:
    """Load label order from an attached Perch artifact.

    Args:
        artifact_dir (Path): Directory containing `labels.json`.

    Returns:
        list[str]: Labels ordered by model output index.
    """
    labels_path = artifact_dir / "labels.json"
    idx_to_label_raw = json.loads(labels_path.read_text(encoding="utf-8"))
    return [idx_to_label_raw[str(i)] for i in range(len(idx_to_label_raw))]


if CFG.mode == "train":
    x = embeddings.astype(np.float32)
    y = train["target"].to_numpy(dtype=np.int64)

    def safe_train_valid_split(
        targets: np.ndarray,
        test_size: float = 0.2,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Create a stratified split while keeping singleton classes in train.

        Args:
            targets (np.ndarray): Integer class targets.
            test_size (float): Validation fraction for non-singleton classes.

        Returns:
            tuple[np.ndarray, np.ndarray]: Train and validation row indices.
        """
        counts = pd.Series(targets).value_counts()
        rare_classes = set(counts[counts < 2].index)
        all_idx = np.arange(len(targets))
        rare_idx = np.array(
            [idx for idx in all_idx if targets[idx] in rare_classes],
            dtype=np.int64,
        )
        common_idx = np.array(
            [idx for idx in all_idx if targets[idx] not in rare_classes],
            dtype=np.int64,
        )

        train_common, valid_idx = train_test_split(
            common_idx,
            test_size=test_size,
            random_state=CFG.seed,
            stratify=targets[common_idx],
        )
        train_idx = np.concatenate([train_common, rare_idx])
        return train_idx, valid_idx

    train_idx, valid_idx = safe_train_valid_split(y)
    valid_meta = train.iloc[valid_idx][
        ["filename", "primary_label", "target"]
    ].reset_index(drop=True)
    print(f"Probe train rows: {len(train_idx):,}")
    print(f"Probe valid rows: {len(valid_idx):,}")
    rare_count = int((pd.Series(y).value_counts() < 2).sum())
    print(f"Classes with fewer than 2 rows kept in train only: {rare_count:,}")

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(x[train_idx]), torch.from_numpy(y[train_idx])),
        batch_size=CFG.probe_batch_size,
        shuffle=True,
    )
    valid_loader = DataLoader(
        TensorDataset(torch.from_numpy(x[valid_idx]), torch.from_numpy(y[valid_idx])),
        batch_size=CFG.probe_batch_size * 2,
        shuffle=False,
    )

    model = PerchProbe(embedding_dim=x.shape[1], num_classes=len(labels)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr)
else:
    if attached_artifact_dir is None:
        raise FileNotFoundError(
            "Submission mode requires an attached Perch artifact directory."
        )
    labels = load_artifact_labels(attached_artifact_dir)
    label_to_idx = {label: idx for idx, label in enumerate(labels)}
    idx_to_label = {idx: label for label, idx in label_to_idx.items()}
    checkpoint_path = attached_artifact_dir / "best_perch_probe.pt"
    checkpoint = torch_load_checkpoint(checkpoint_path)
    model = PerchProbe(
        embedding_dim=CFG.embedding_dim,
        num_classes=len(labels),
    ).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    print(f"Loaded probe checkpoint: {checkpoint_path}")
    print(f"Loaded labels: {len(labels):,}")


Loaded probe checkpoint: /kaggle/input/models/tuannm3812/birdclef-perch-v2-artifacts/pytorch/default/1/perch_v2/best_perch_probe.pt
Loaded labels: 206


## 6. Training


Rare singleton classes stay in training so the stratified validation split remains valid. The training loop tracks **best validation accuracy**, applies **early stopping**, and writes diagnostic files for per-class analysis.


In [6]:
if CFG.mode == "train":
    @torch.no_grad()
    def evaluate(loader: DataLoader) -> tuple[float, float]:
        """Evaluate probe loss and top-1 accuracy.

        Args:
            loader (DataLoader): Validation DataLoader.

        Returns:
            tuple[float, float]: Mean loss and top-1 accuracy.
        """
        model.eval()
        total_loss = 0.0
        correct = 0
        seen = 0
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            seen += xb.size(0)
        return total_loss / max(seen, 1), correct / max(seen, 1)


    @torch.no_grad()
    def collect_validation_predictions() -> tuple[pd.DataFrame, pd.DataFrame, float]:
        """Collect row-level and per-class validation diagnostics.

        Returns:
            tuple[pd.DataFrame, pd.DataFrame, float]: Row predictions,
            per-class metrics, and top-5 accuracy.
        """
        model.eval()
        top1_labels, top1_probs, top5_labels = [], [], []
        top5_hits = []

        for xb, yb in valid_loader:
            xb = xb.to(device)
            probs = torch.softmax(model(xb), dim=1).cpu()
            values, indices = probs.topk(k=min(5, probs.shape[1]), dim=1)
            for row_values, row_indices, target in zip(values, indices, yb):
                labels_top = [idx_to_label[int(idx)] for idx in row_indices]
                top1_labels.append(labels_top[0])
                top1_probs.append(float(row_values[0]))
                top5_labels.append(" ".join(labels_top))
                top5_hits.append(int(int(target) in [int(idx) for idx in row_indices]))

        pred_df = valid_meta.copy()
        pred_df["top1_label"] = top1_labels
        pred_df["top1_prob"] = top1_probs
        pred_df["top5_labels"] = top5_labels
        pred_df["top1_correct"] = pred_df["primary_label"].eq(pred_df["top1_label"]).astype(int)
        pred_df["top5_correct"] = top5_hits

        class_df = (
            pred_df.groupby("primary_label")
            .agg(
                support=("filename", "count"),
                top1_recall=("top1_correct", "mean"),
                top5_recall=("top5_correct", "mean"),
                mean_top1_prob=("top1_prob", "mean"),
            )
            .reset_index()
            .sort_values(["top1_recall", "support"], ascending=[True, False])
        )
        return pred_df, class_df, float(pred_df["top5_correct"].mean())


    history = []
    best_acc = 0.0
    epochs_without_improvement = 0

    for epoch in range(1, CFG.probe_epochs + 1):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)

        valid_loss, valid_acc = evaluate(valid_loader)
        row = {
            "epoch": epoch,
            "train_loss": total_loss / max(len(train_loader.dataset), 1),
            "valid_loss": valid_loss,
            "valid_acc": valid_acc,
        }
        history.append(row)
        print(row)

        improved = valid_acc > best_acc + CFG.early_stopping_min_delta
        if improved:
            best_acc = valid_acc
            epochs_without_improvement = 0
            torch.save(
                {
                    "model": model.state_dict(),
                    "label_to_idx": label_to_idx,
                    "cfg": {k: v for k, v in CFG.__dict__.items() if not k.startswith("_")},
                    "valid_acc": best_acc,
                },
                CFG.artifact_dir / "best_perch_probe.pt",
            )
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= CFG.early_stopping_patience:
                print(f"Early stopping after {epoch} epochs; best valid accuracy: {best_acc:.4f}")
                break

    history_df = pd.DataFrame(history)
    history_df.to_csv(CFG.artifact_dir / "history.csv", index=False)

    try:
        best_checkpoint = torch.load(CFG.artifact_dir / "best_perch_probe.pt", map_location=device, weights_only=False)
    except TypeError:
        best_checkpoint = torch.load(CFG.artifact_dir / "best_perch_probe.pt", map_location=device)
    model.load_state_dict(best_checkpoint["model"])
    validation_predictions, per_class_metrics, valid_top5_acc = collect_validation_predictions()
    validation_predictions.to_csv(CFG.artifact_dir / "validation_predictions.csv", index=False)
    per_class_metrics.to_csv(CFG.artifact_dir / "per_class_validation_metrics.csv", index=False)
    summary = {
        "best_valid_acc": float(best_acc),
        "valid_top5_acc": valid_top5_acc,
        "epochs_ran": len(history_df),
        "best_epoch": int(history_df.loc[history_df["valid_acc"].idxmax(), "epoch"]),
        "embedding_shape": list(embeddings.shape),
    }
    (CFG.artifact_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print(f"Best valid accuracy: {best_acc:.4f}")
    print(f"Validation top-5 accuracy: {valid_top5_acc:.4f}")
    print(f"Saved outputs to {CFG.artifact_dir}")
    display(per_class_metrics.head(10))
else:
    print("Submission mode: skipped probe training and diagnostics.")


Submission mode: skipped probe training and diagnostics.


## 7. Submission

Load `sample_submission.csv`, extract Perch embeddings for hidden-test windows, score them with the attached probe checkpoint, and write `/kaggle/working/submission.csv`.


In [7]:
if CFG.mode == "submission":
    def row_to_stem_and_end_time(row_id: str) -> tuple[str, float]:
        """Parse a submission row id into audio stem and window end time.

        Args:
            row_id (str): Submission row id.

        Returns:
            tuple[str, float]: Audio stem and segment end time in seconds.
        """
        stem, sep, end = str(row_id).rpartition("_")
        if sep and end.replace(".", "", 1).isdigit():
            return stem, float(end)
        return str(row_id), CFG.duration


    def build_audio_index() -> dict[str, Path]:
        """Index hidden-test audio files by filename stem.

        Returns:
            dict[str, Path]: Mapping from audio stem to file path.
        """
        audio_index = {}
        for folder in [CFG.data_root / "test_soundscapes", CFG.data_root / "test_audio"]:
            if folder.exists():
                for ext in ("*.ogg", "*.wav", "*.flac", "*.mp3"):
                    for path in folder.glob(ext):
                        audio_index[path.stem] = path
        return audio_index


    @torch.no_grad()
    def predict_submission_batch(batch_waveforms: list[np.ndarray]) -> np.ndarray:
        """Predict class probabilities for waveform segments.

        Args:
            batch_waveforms (list[np.ndarray]): Waveform segments.

        Returns:
            np.ndarray: Class probabilities ordered by `labels`.
        """
        embeddings = run_perch_batch(np.stack(batch_waveforms))
        x = torch.from_numpy(embeddings).to(device)
        logits = model(x)
        return torch.softmax(logits, dim=1).cpu().numpy()


    sample_path = CFG.data_root / "sample_submission.csv"
    sample_submission = pd.read_csv(sample_path)
    target_columns = [col for col in sample_submission.columns if col != "row_id"]
    target_positions = [idx for idx, label in enumerate(labels) if label in target_columns]
    target_labels = [labels[idx] for idx in target_positions]
    audio_index = build_audio_index()

    print(f"Sample submission: {sample_path}")
    print(f"Rows: {len(sample_submission):,}")
    print(f"Indexed audio files: {len(audio_index):,}")
    print(f"Submission target columns: {len(target_columns):,}")
    print(f"Model target overlap: {len(target_labels):,}")

    submission = sample_submission.copy()
    for col in target_columns:
        submission[col] = 0.0

    if len(audio_index) == 0 and len(submission) <= 3:
        print("Tiny public dry-run detected with no test audio; preserving sample submission values.")
        submission = sample_submission.copy()
    else:
        batch, batch_rows = [], []
        iterator = tqdm(list(enumerate(submission["row_id"])), desc="inference")
        for row_idx, row_id in iterator:
            stem, end_time = row_to_stem_and_end_time(row_id)
            audio_path = audio_index.get(stem)
            if audio_path is None:
                raise FileNotFoundError(
                    f"Missing audio for row_id={row_id}; expected stem={stem}"
                )
            batch.append(load_audio_segment(audio_path, end_time))
            batch_rows.append(row_idx)
            if len(batch) == CFG.inference_batch_size:
                probs = predict_submission_batch(batch)[:, target_positions]
                submission.loc[batch_rows, target_labels] = probs
                batch, batch_rows = [], []

        if batch:
            probs = predict_submission_batch(batch)[:, target_positions]
            submission.loc[batch_rows, target_labels] = probs

    submission_path = Path("/kaggle/working/submission.csv")
    submission.to_csv(submission_path, index=False)
    print(f"Wrote submission: {submission_path}")
    display(submission.head())
else:
    print("Train mode: skipped submission inference.")


Sample submission: /kaggle/input/competitions/birdclef-2026/sample_submission.csv
Rows: 3
Indexed audio files: 0
Submission target columns: 234
Model target overlap: 206
Tiny public dry-run detected with no test audio; preserving sample submission values.
Wrote submission: /kaggle/working/submission.csv


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,22967,22973,22983,22985,23150,23154,23158,23176,23724,24279,24285,24287,24321,244024,25073,25092,25214,326272,41970,43435,47144,47158son01,47158son02,47158son03,47158son04,47158son05,47158son06,47158son07,47158son08,47158son09,...,sobtyr1,socfly1,sofspi1,souant1,soulap1,souscr1,spbant3,spispi1,sptnig1,squcuc1,stbwoo2,strcuc1,strher2,strowl1,swthum1,swtman1,tattin1,thlwre1,toctou1,trokin,trsowl,undtin1,varant1,watjac1,wesfie1,wfwduc1,whbant2,whbwar2,whiwoo1,whlspi1,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


## 8. Interpretation

- **Perch embeddings** provide the strongest validation signal in this repo and are best used as teacher features or distillation targets.
- **Early stopping** is important: validation peaks quickly while train loss keeps falling, which suggests the probe can overfit frozen features.
- **Per-class recall** and **top-5 hit rate** identify which labels are already well represented by Perch and which labels still need sampling, augmentation, or domain-specific thresholds.
- Direct Perch inference is operationally heavier than EfficientNet, so keep it experimental until hidden-test runtime is proven.


## 9. Artifacts


Package the **embedding cache**, **best probe checkpoint**, **training history**, **validation predictions**, and **per-class metrics** for Kaggle download. Large generated artifacts remain outside Git.


In [8]:
import zipfile
from IPython.display import FileLink

if CFG.mode == "train":
    zip_path = Path("/kaggle/working/birdclef_perch_v2_artifacts.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(CFG.artifact_dir.rglob("*")):
            if path.is_file():
                zf.write(path, arcname=path.relative_to(CFG.artifact_dir.parent))

    print(f"Wrote {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
    display(FileLink(zip_path))
else:
    print("Submission mode: skipped training artifact zip.")


Submission mode: skipped training artifact zip.
